In [56]:
import pandas as pd
import numpy as np
import logging
from pathlib import Path
from datetime import datetime

monthly_kpi = pd.read_csv("/content/monthly_kpi_summary.csv")
channel_performance = pd.read_csv("/content/channel_performance.csv")

raw_campaigns = pd.read_csv("/content/raw_campaigns.csv")
raw_customers = pd.read_csv("/content/raw_customers.csv")
raw_conversions = pd.read_csv("/content/raw_conversions.csv")



In [57]:
datasets = {
    "Monthly KPI Summary": monthly_kpi,
    "Channel Performance": channel_performance,
    "Raw Campaigns": raw_campaigns,
    "Raw Customers": raw_customers,
    "Raw Conversions": raw_conversions
}

for name, dataframe in datasets.items():
    print(f"{name}: {dataframe.shape[0]} rows, {dataframe.shape[1]} columns")

Monthly KPI Summary: 12 rows, 14 columns
Channel Performance: 7 rows, 14 columns
Raw Campaigns: 200 rows, 12 columns
Raw Customers: 10000 rows, 10 columns
Raw Conversions: 40005 rows, 12 columns


In [58]:
monthly_kpi["report_month"] = pd.to_datetime(
    monthly_kpi["report_month"],
    errors="coerce"
)

monthly_kpi = monthly_kpi.sort_values("report_month").reset_index(drop=True)

display(monthly_kpi.head())

,report_month,total_campaigns,total_marketing_spend,total_impressions,total_clicks,total_conversions,new_customers,total_revenue,average_order_value,ctr_percentage,conversion_rate_percentage,customer_acquisition_cost,roas,cost_per_click
0,2025-01-01,17,1345125.0,6280197,188767,2899,647,4576783.97,1810.88,3.01,1.54,2079.02,3.40,7.13
1,2025-02-01,17,1310764.0,6288003,222402,3013,681,5064629.74,1888.36,3.54,1.35,1924.76,3.86,5.89
2,2025-03-01,17,1314868.0,6778778,252586,3039,692,5096502.64,1912.32,3.73,1.20,1900.10,3.88,5.21
3,2025-04-01,17,1178049.0,7419426,241276,2953,663,4796561.88,1840.19,3.25,1.22,1776.85,4.07,4.88
4,2025-05-01,17,917719.0,5010443,185918,3004,718,4972700.54,1881.17,3.71,1.62,1278.16,5.42,4.94


In [59]:
validation_results = []

for name, dataframe in datasets.items():
    validation_results.append({
        "dataset": name,
        "row_count": len(dataframe),
        "column_count": len(dataframe.columns),
        "duplicate_rows": dataframe.duplicated().sum(),
        "missing_values": dataframe.isna().sum().sum(),
        "status": "Passed" if len(dataframe) > 0 else "Failed"
    })

validation_summary = pd.DataFrame(validation_results)

display(validation_summary)

,dataset,row_count,column_count,duplicate_rows,missing_values,status
0,Monthly KPI Summary,12,14,0,0,Passed
1,Channel Performance,7,14,0,0,Passed
2,Raw Campaigns,200,12,0,1,Passed
3,Raw Customers,10000,10,0,1,Passed
4,Raw Conversions,40005,12,5,1,Passed


In [60]:
required_kpi_columns = [
    "report_month",
    "total_marketing_spend",
    "total_impressions",
    "total_clicks",
    "total_conversions",
    "new_customers",
    "total_revenue",
    "ctr_percentage",
    "conversion_rate_percentage",
    "customer_acquisition_cost",
    "roas",
    "cost_per_click",
    "average_order_value"
]

missing_columns = [
    column
    for column in required_kpi_columns
    if column not in monthly_kpi.columns
]

if missing_columns:
    print("Missing KPI columns:", missing_columns)
else:
    print("All required KPI columns are available.")

All required KPI columns are available.


In [61]:
invalid_months = monthly_kpi[
    (monthly_kpi["report_month"] < "2025-01-01") |
    (monthly_kpi["report_month"] > "2025-12-31")
]

if invalid_months.empty:
    print("Reporting period validation passed.")
else:
    print("Invalid reporting months found:")
    display(invalid_months)

Reporting period validation passed.


In [62]:
monthly_kpi = monthly_kpi.sort_values(
    "report_month"
).reset_index(drop=True)

display(monthly_kpi.head())

,report_month,total_campaigns,total_marketing_spend,total_impressions,total_clicks,total_conversions,new_customers,total_revenue,average_order_value,ctr_percentage,conversion_rate_percentage,customer_acquisition_cost,roas,cost_per_click
0,2025-01-01,17,1345125.0,6280197,188767,2899,647,4576783.97,1810.88,3.01,1.54,2079.02,3.40,7.13
1,2025-02-01,17,1310764.0,6288003,222402,3013,681,5064629.74,1888.36,3.54,1.35,1924.76,3.86,5.89
2,2025-03-01,17,1314868.0,6778778,252586,3039,692,5096502.64,1912.32,3.73,1.20,1900.10,3.88,5.21
3,2025-04-01,17,1178049.0,7419426,241276,2953,663,4796561.88,1840.19,3.25,1.22,1776.85,4.07,4.88
4,2025-05-01,17,917719.0,5010443,185918,3004,718,4972700.54,1881.17,3.71,1.62,1278.16,5.42,4.94


In [63]:
mom_kpis = [
    "total_marketing_spend",
    "total_conversions",
    "new_customers",
    "total_revenue",
    "ctr_percentage",
    "conversion_rate_percentage",
    "customer_acquisition_cost",
    "roas",
    "cost_per_click",
    "average_order_value"
]

In [64]:
for column in mom_kpis:
    monthly_kpi[f"previous_{column}"] = monthly_kpi[column].shift(1)

In [65]:
for column in mom_kpis:
    monthly_kpi[f"{column}_mom_pct"] = (
        (
            monthly_kpi[column]
            - monthly_kpi[f"previous_{column}"]
        )
        / monthly_kpi[f"previous_{column}"].replace(0, np.nan)
        * 100
    ).round(2)

In [66]:
mom_summary = monthly_kpi[
    [
        "report_month",
        "total_marketing_spend",
        "total_marketing_spend_mom_pct",
        "total_revenue",
        "total_revenue_mom_pct",
        "total_conversions",
        "total_conversions_mom_pct",
        "customer_acquisition_cost",
        "customer_acquisition_cost_mom_pct",
        "roas",
        "roas_mom_pct"
    ]
]

display(mom_summary)

,report_month,total_marketing_spend,total_marketing_spend_mom_pct,total_revenue,total_revenue_mom_pct,total_conversions,total_conversions_mom_pct,customer_acquisition_cost,customer_acquisition_cost_mom_pct,roas,roas_mom_pct
0,2025-01-01,1345125.0,NaN,4576783.97,NaN,2899,NaN,2079.02,NaN,3.40,NaN
1,2025-02-01,1310764.0,-2.55,5064629.74,10.66,3013,3.93,1924.76,-7.42,3.86,13.53
2,2025-03-01,1314868.0,0.31,5096502.64,0.63,3039,0.86,1900.10,-1.28,3.88,0.52
3,2025-04-01,1178049.0,-10.41,4796561.88,-5.89,2953,-2.83,1776.85,-6.49,4.07,4.90
4,2025-05-01,917719.0,-22.10,4972700.54,3.67,3004,1.73,1278.16,-28.07,5.42,33.17
5,2025-06-01,1285880.0,40.12,4870461.24,-2.06,3041,1.23,1705.41,33.43,3.79,-30.07
6,2025-07-01,1337388.0,4.01,5024211.17,3.16,3054,0.43,1852.34,8.62,3.76,-0.79
7,2025-08-01,1377777.0,3.02,5101544.18,1.54,2954,-3.27,2044.18,10.36,3.70,-1.60
8,2025-09-01,1117580.0,-18.89,4626714.68,-9.31,2778,-5.96,1762.74,-13.77,4.14,11.89
9,2025-10-01,1210884.0,8.35,4669639.98,0.93,2775,-0.11,1734.79,-1.59,3.86,-6.76


In [67]:
monthly_kpi["month_name"] = monthly_kpi[
    "report_month"
].dt.strftime("%b %Y")

In [68]:
monthly_kpi["revenue_direction"] = np.where(
    monthly_kpi["total_revenue_mom_pct"] > 0,
    "Improved",
    np.where(
        monthly_kpi["total_revenue_mom_pct"] < 0,
        "Declined",
        "No Change"
    )
)

In [69]:
monthly_kpi["roas_direction"] = np.where(
    monthly_kpi["roas_mom_pct"] > 0,
    "Improved",
    np.where(
        monthly_kpi["roas_mom_pct"] < 0,
        "Declined",
        "No Change"
    )
)

In [70]:
monthly_kpi["cac_direction"] = np.where(
    monthly_kpi["customer_acquisition_cost_mom_pct"] < 0,
    "Improved",
    np.where(
        monthly_kpi["customer_acquisition_cost_mom_pct"] > 0,
        "Declined",
        "No Change"
    )
)

In [71]:
direction_columns = [
    "revenue_direction",
    "roas_direction",
    "cac_direction"
]

monthly_kpi.loc[0, direction_columns] = "No Previous Month"


In [72]:
best_revenue_month = monthly_kpi.loc[
    monthly_kpi["total_revenue"].idxmax(),
    ["month_name", "total_revenue"]
]

print("Highest Revenue Month")
print(best_revenue_month)

Highest Revenue Month
month_name         Aug 2025
total_revenue    5101544.18
Name: 7, dtype: object


In [73]:
best_roas_month = monthly_kpi.loc[
    monthly_kpi["roas"].idxmax(),
    ["month_name", "roas"]
]

print("Highest ROAS Month")
print(best_roas_month)

Highest ROAS Month
month_name    May 2025
roas              5.42
Name: 4, dtype: object


In [74]:
valid_cac = monthly_kpi[
    monthly_kpi["customer_acquisition_cost"].notna()
]

best_cac_month = valid_cac.loc[
    valid_cac["customer_acquisition_cost"].idxmin(),
    ["month_name", "customer_acquisition_cost"]
]

print("Lowest CAC Month")
print(best_cac_month)

Lowest CAC Month
month_name                   May 2025
customer_acquisition_cost     1278.16
Name: 4, dtype: object


In [75]:
def assign_monthly_status(row):
    if (
        row["roas"] >= 4
        and row["total_revenue_mom_pct"] > 0
        and row["customer_acquisition_cost_mom_pct"] <= 0
    ):
        return "High Performing"

    elif (
        row["roas"] >= 2
        and row["total_revenue_mom_pct"] >= -5
    ):
        return "Stable"

    else:
        return "Needs Attention"

In [76]:
monthly_kpi["performance_status"] = monthly_kpi.apply(
    assign_monthly_status,
    axis=1
)

monthly_kpi.loc[0, "performance_status"] = "Baseline Month"

In [77]:
monthly_trends = monthly_kpi[
    [
        "report_month",
        "month_name",
        "total_marketing_spend",
        "total_marketing_spend_mom_pct",
        "total_revenue",
        "total_revenue_mom_pct",
        "total_conversions",
        "total_conversions_mom_pct",
        "customer_acquisition_cost",
        "customer_acquisition_cost_mom_pct",
        "roas",
        "roas_mom_pct",
        "revenue_direction",
        "roas_direction",
        "cac_direction",
        "performance_status"
    ]
]

display(monthly_trends)

,report_month,month_name,total_marketing_spend,total_marketing_spend_mom_pct,total_revenue,total_revenue_mom_pct,total_conversions,total_conversions_mom_pct,customer_acquisition_cost,customer_acquisition_cost_mom_pct,roas,roas_mom_pct,revenue_direction,roas_direction,cac_direction,performance_status
0,2025-01-01,Jan 2025,1345125.0,NaN,4576783.97,NaN,2899,NaN,2079.02,NaN,3.40,NaN,No Previous Month,No Previous Month,No Previous Month,Baseline Month
1,2025-02-01,Feb 2025,1310764.0,-2.55,5064629.74,10.66,3013,3.93,1924.76,-7.42,3.86,13.53,Improved,Improved,Improved,Stable
2,2025-03-01,Mar 2025,1314868.0,0.31,5096502.64,0.63,3039,0.86,1900.10,-1.28,3.88,0.52,Improved,Improved,Improved,Stable
3,2025-04-01,Apr 2025,1178049.0,-10.41,4796561.88,-5.89,2953,-2.83,1776.85,-6.49,4.07,4.90,Declined,Improved,Improved,Needs Attention
4,2025-05-01,May 2025,917719.0,-22.10,4972700.54,3.67,3004,1.73,1278.16,-28.07,5.42,33.17,Improved,Improved,Improved,High Performing
5,2025-06-01,Jun 2025,1285880.0,40.12,4870461.24,-2.06,3041,1.23,1705.41,33.43,3.79,-30.07,Declined,Declined,Declined,Stable
6,2025-07-01,Jul 2025,1337388.0,4.01,5024211.17,3.16,3054,0.43,1852.34,8.62,3.76,-0.79,Improved,Declined,Declined,Stable
7,2025-08-01,Aug 2025,1377777.0,3.02,5101544.18,1.54,2954,-3.27,2044.18,10.36,3.70,-1.60,Improved,Declined,Declined,Stable
8,2025-09-01,Sep 2025,1117580.0,-18.89,4626714.68,-9.31,2778,-5.96,1762.74,-13.77,4.14,11.89,Declined,Improved,Improved,Needs Attention
9,2025-10-01,Oct 2025,1210884.0,8.35,4669639.98,0.93,2775,-0.11,1734.79,-1.59,3.86,-6.76,Improved,Declined,Improved,Stable


In [78]:
monthly_trends.to_csv(
    "/content/monthly_trends.csv",
    index=False
)


In [79]:
def add_quality_check(dataset, check_name, issue_count, comments):
    data_quality_results.append({
        "dataset": dataset,
        "check_name": check_name,
        "issue_count": int(issue_count),
        "status": "Passed" if issue_count == 0 else "Failed",
        "comments": comments
    })

In [80]:
data_quality_results = []
duplicate_campaign_ids = raw_campaigns["Campaign_ID"].duplicated().sum()

add_quality_check(
    "campaigns",
    "Duplicate Campaign IDs",
    duplicate_campaign_ids,
    "Campaign_ID should be unique."
)

In [81]:
missing_campaign_channels = raw_campaigns["Channel"].isna().sum()

add_quality_check(
    "campaigns",
    "Missing Campaign Channels",
    missing_campaign_channels,
    "Channel should not be missing."
)

In [82]:
negative_campaign_spend = (
    raw_campaigns["Marketing_Spend"] < 0
).sum()

add_quality_check(
    "campaigns",
    "Negative Marketing Spend",
    negative_campaign_spend,
    "Marketing spend cannot be negative."
)

In [83]:

invalid_clicks = (
    raw_campaigns["Clicks"]
    > raw_campaigns["Impressions"]
).sum()

add_quality_check(
    "campaigns",
    "Clicks Greater Than Impressions",
    invalid_clicks,
    "Clicks should not exceed impressions."
)

In [84]:
raw_campaigns["Start_Date"] = pd.to_datetime(
    raw_campaigns["Start_Date"],
    errors="coerce"
)

raw_campaigns["End_Date"] = pd.to_datetime(
    raw_campaigns["End_Date"],
    errors="coerce"
)

invalid_campaign_dates = (
    raw_campaigns["End_Date"]
    < raw_campaigns["Start_Date"]
).sum()

add_quality_check(
    "campaigns",
    "End Date Before Start Date",
    invalid_campaign_dates,
    "Campaign end date must be after the start date."
)

In [85]:
duplicate_customer_ids = raw_customers[
    "Customer_ID"
].duplicated().sum()

add_quality_check(
    "customers",
    "Duplicate Customer IDs",
    duplicate_customer_ids,
    "Customer_ID should be unique."
)

In [86]:
missing_customer_segments = raw_customers[
    "Customer_Segment"
].isna().sum()

add_quality_check(
    "customers",
    "Missing Customer Segments",
    missing_customer_segments,
    "Customer segment should not be missing."
)

In [87]:
negative_lifetime_value = (
    raw_customers["Lifetime_Value"] < 0
).sum()

add_quality_check(
    "customers",
    "Negative Lifetime Value",
    negative_lifetime_value,
    "Lifetime value cannot be negative."
)

In [88]:
negative_previous_orders = (
    raw_customers["Total_Previous_Orders"] < 0
).sum()

add_quality_check(
    "customers",
    "Negative Previous Orders",
    negative_previous_orders,
    "Previous order count cannot be negative."
)

In [89]:
valid_channels = [
    "Email",
    "Paid Search",
    "Social Media",
    "Display Ads",
    "Affiliate",
    "Referral"
]

invalid_acquisition_channels = (
    ~raw_customers["Acquisition_Channel"].isin(valid_channels)
    & raw_customers["Acquisition_Channel"].notna()
).sum()

add_quality_check(
    "customers",
    "Invalid Acquisition Channels",
    invalid_acquisition_channels,
    "Acquisition channel should match the approved channel list."
)

In [90]:

duplicate_conversion_ids = raw_conversions[
    "Conversion_ID"
].duplicated().sum()

add_quality_check(
    "conversions",
    "Duplicate Conversion IDs",
    duplicate_conversion_ids,
    "Conversion_ID should be unique."
)

In [91]:
missing_payment_methods = raw_conversions[
    "Payment_Method"
].isna().sum()

add_quality_check(
    "conversions",
    "Missing Payment Methods",
    missing_payment_methods,
    "Payment method should not be missing."
)

In [92]:

negative_revenue = (
    raw_conversions["Revenue"] < 0
).sum()

add_quality_check(
    "conversions",
    "Negative Revenue",
    negative_revenue,
    "Revenue cannot be negative."
)

In [93]:
negative_order_value = (
    raw_conversions["Order_Value"] < 0
).sum()

add_quality_check(
    "conversions",
    "Negative Order Value",
    negative_order_value,
    "Order value cannot be negative."
)

In [94]:
raw_conversions["Conversion_Date"] = pd.to_datetime(
    raw_conversions["Conversion_Date"],
    errors="coerce"
)

invalid_conversion_dates = (
    (raw_conversions["Conversion_Date"] < "2025-01-01")
    |
    (raw_conversions["Conversion_Date"] > "2025-12-31")
).sum()

add_quality_check(
    "conversions",
    "Conversions Outside Reporting Period",
    invalid_conversion_dates,
    "Conversion dates must fall within 2025."
)

In [95]:
valid_campaign_ids = set(
    raw_campaigns["Campaign_ID"].dropna()
)

invalid_campaign_references = (
    ~raw_conversions["Campaign_ID"].isin(valid_campaign_ids)
).sum()

add_quality_check(
    "conversions",
    "Invalid Campaign References",
    invalid_campaign_references,
    "Campaign_ID must exist in the campaigns dataset."
)

In [96]:
valid_customer_ids = set(
    raw_customers["Customer_ID"].dropna()
)

invalid_customer_references = (
    ~raw_conversions["Customer_ID"].isin(valid_customer_ids)
).sum()

add_quality_check(
    "conversions",
    "Invalid Customer References",
    invalid_customer_references,
    "Customer_ID must exist in the customers dataset."
)

In [97]:
invalid_completed_purchases = raw_conversions[
    (raw_conversions["Conversion_Status"] == "Completed")
    &
    (raw_conversions["Conversion_Type"] == "Purchase")
    &
    (raw_conversions["Revenue"] <= 0)
].shape[0]

add_quality_check(
    "conversions",
    "Completed Purchases With Invalid Revenue",
    invalid_completed_purchases,
    "Completed purchases should have positive revenue."
)

In [98]:
data_quality_report = pd.DataFrame(
    data_quality_results
)

data_quality_report["check_date"] = datetime.now().strftime(
    "%Y-%m-%d %H:%M:%S"
)

display(data_quality_report)

,dataset,check_name,issue_count,status,comments,check_date
0,campaigns,Duplicate Campaign IDs,0,Passed,Campaign_ID should be unique.,2026-06-22 11:44:17
1,campaigns,Missing Campaign Channels,1,Failed,Channel should not be missing.,2026-06-22 11:44:17
2,campaigns,Negative Marketing Spend,0,Passed,Marketing spend cannot be negative.,2026-06-22 11:44:17
3,campaigns,Clicks Greater Than Impressions,1,Failed,Clicks should not exceed impressions.,2026-06-22 11:44:17
4,campaigns,End Date Before Start Date,0,Passed,Campaign end date must be after the start date.,2026-06-22 11:44:17
5,customers,Duplicate Customer IDs,0,Passed,Customer_ID should be unique.,2026-06-22 11:44:17
6,customers,Missing Customer Segments,1,Failed,Customer segment should not be missing.,2026-06-22 11:44:17
7,customers,Negative Lifetime Value,1,Failed,Lifetime value cannot be negative.,2026-06-22 11:44:17
8,customers,Negative Previous Orders,0,Passed,Previous order count cannot be negative.,2026-06-22 11:44:17
9,customers,Invalid Acquisition Channels,1,Failed,Acquisition channel should match the approved ...,2026-06-22 11:44:17


In [99]:
total_checks = len(data_quality_report)

passed_checks = (
    data_quality_report["status"] == "Passed"
).sum()

failed_checks = (
    data_quality_report["status"] == "Failed"
).sum()

total_issues = data_quality_report[
    "issue_count"
].sum()

print("Total Checks:", total_checks)
print("Passed Checks:", passed_checks)
print("Failed Checks:", failed_checks)
print("Total Issues Found:", total_issues)

Total Checks: 18
Passed Checks: 6
Failed Checks: 12
Total Issues Found: 16


In [100]:
data_quality_score = round(
    passed_checks / total_checks * 100,
    2
)

print(f"Data Quality Score: {data_quality_score}%")

Data Quality Score: 33.33%


In [101]:
failed_quality_checks = data_quality_report[
    data_quality_report["status"] == "Failed"
]

display(failed_quality_checks)

,dataset,check_name,issue_count,status,comments,check_date
1,campaigns,Missing Campaign Channels,1,Failed,Channel should not be missing.,2026-06-22 11:44:17
3,campaigns,Clicks Greater Than Impressions,1,Failed,Clicks should not exceed impressions.,2026-06-22 11:44:17
6,customers,Missing Customer Segments,1,Failed,Customer segment should not be missing.,2026-06-22 11:44:17
7,customers,Negative Lifetime Value,1,Failed,Lifetime value cannot be negative.,2026-06-22 11:44:17
9,customers,Invalid Acquisition Channels,1,Failed,Acquisition channel should match the approved ...,2026-06-22 11:44:17
10,conversions,Duplicate Conversion IDs,5,Failed,Conversion_ID should be unique.,2026-06-22 11:44:17
11,conversions,Missing Payment Methods,1,Failed,Payment method should not be missing.,2026-06-22 11:44:17
12,conversions,Negative Revenue,1,Failed,Revenue cannot be negative.,2026-06-22 11:44:17
14,conversions,Conversions Outside Reporting Period,1,Failed,Conversion dates must fall within 2025.,2026-06-22 11:44:17
15,conversions,Invalid Campaign References,1,Failed,Campaign_ID must exist in the campaigns dataset.,2026-06-22 11:44:17


In [102]:
data_quality_report.to_csv(
    "/content/data_quality_report.csv",
    index=False
)

print("data_quality_report.csv created successfully.")

data_quality_report.csv created successfully.


In [103]:
failed_quality_checks.to_csv(
    "/content/failed_quality_checks.csv",
    index=False
)



In [104]:
from pathlib import Path

output_folder = Path("/content/outputs")
log_folder = Path("/content/logs")

output_folder.mkdir(exist_ok=True)
log_folder.mkdir(exist_ok=True)

print("Output and log folders created.")

Output and log folders created.


In [105]:
import logging
from datetime import datetime

log_file = log_folder / "reporting_pipeline.log"

logging.basicConfig(
    filename=log_file,
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    filemode="w"
)

logging.info("Monthly reporting pipeline started.")

In [106]:
import pandas as pd

def load_csv_file(file_path, file_name):
    try:
        dataframe = pd.read_csv(file_path)

        logging.info(
            f"{file_name} loaded successfully: "
            f"{len(dataframe)} rows and "
            f"{len(dataframe.columns)} columns"
        )

        print(f"{file_name} loaded successfully.")

        return dataframe

    except FileNotFoundError:
        logging.error(f"{file_name} was not found.")
        raise

    except Exception as error:
        logging.error(
            f"Error while loading {file_name}: {error}"
        )
        raise

In [107]:
monthly_kpi = load_csv_file(
    "/content/monthly_kpi_summary.csv",
    "Monthly KPI Summary"
)

channel_performance = load_csv_file(
    "/content/channel_performance.csv",
    "Channel Performance"
)

raw_campaigns = load_csv_file(
    "/content/raw_campaigns.csv",
    "Raw Campaigns"
)

raw_customers = load_csv_file(
    "/content/raw_customers.csv",
    "Raw Customers"
)

raw_conversions = load_csv_file(
    "/content/raw_conversions.csv",
    "Raw Conversions"
)

Monthly KPI Summary loaded successfully.
Channel Performance loaded successfully.
Raw Campaigns loaded successfully.
Raw Customers loaded successfully.
Raw Conversions loaded successfully.


In [108]:
def validate_required_columns(
    dataframe,
    required_columns,
    dataset_name
):
    missing_columns = [
        column
        for column in required_columns
        if column not in dataframe.columns
    ]

    if missing_columns:
        logging.error(
            f"{dataset_name} missing columns: "
            f"{missing_columns}"
        )

        raise ValueError(
            f"{dataset_name} is missing columns: "
            f"{missing_columns}"
        )

    logging.info(
        f"Required column validation passed for {dataset_name}."
    )

    print(
        f"Required column validation passed for {dataset_name}."
    )

In [109]:
required_monthly_columns = [
    "report_month",
    "total_marketing_spend",
    "total_impressions",
    "total_clicks",
    "total_conversions",
    "new_customers",
    "total_revenue",
    "ctr_percentage",
    "conversion_rate_percentage",
    "customer_acquisition_cost",
    "roas",
    "cost_per_click",
    "average_order_value"
]

validate_required_columns(
    monthly_kpi,
    required_monthly_columns,
    "Monthly KPI Summary"
)

Required column validation passed for Monthly KPI Summary.


In [110]:
import numpy as np

def create_monthly_trends(monthly_data):
    monthly_data = monthly_data.copy()

    monthly_data["report_month"] = pd.to_datetime(
        monthly_data["report_month"],
        errors="coerce"
    )

    monthly_data = monthly_data.sort_values(
        "report_month"
    ).reset_index(drop=True)

    mom_columns = [
        "total_marketing_spend",
        "total_conversions",
        "new_customers",
        "total_revenue",
        "customer_acquisition_cost",
        "roas",
        "cost_per_click",
        "average_order_value"
    ]

    for column in mom_columns:
        monthly_data[f"{column}_mom_pct"] = (
            monthly_data[column]
            .pct_change(fill_method=None)
            .replace([np.inf, -np.inf], np.nan)
            .mul(100)
            .round(2)
        )

    monthly_data["month_name"] = monthly_data[
        "report_month"
    ].dt.strftime("%b %Y")

    monthly_data["revenue_direction"] = np.select(
        [
            monthly_data["total_revenue_mom_pct"] > 0,
            monthly_data["total_revenue_mom_pct"] < 0
        ],
        [
            "Improved",
            "Declined"
        ],
        default="No Change"
    )

    monthly_data["roas_direction"] = np.select(
        [
            monthly_data["roas_mom_pct"] > 0,
            monthly_data["roas_mom_pct"] < 0
        ],
        [
            "Improved",
            "Declined"
        ],
        default="No Change"
    )

    monthly_data["cac_direction"] = np.select(
        [
            monthly_data[
                "customer_acquisition_cost_mom_pct"
            ] < 0,

            monthly_data[
                "customer_acquisition_cost_mom_pct"
            ] > 0
        ],
        [
            "Improved",
            "Declined"
        ],
        default="No Change"
    )

    if not monthly_data.empty:
        monthly_data.loc[
            0,
            [
                "revenue_direction",
                "roas_direction",
                "cac_direction"
            ]
        ] = "No Previous Month"

    logging.info(
        "Month-over-month calculations completed."
    )

    return monthly_data

In [111]:
monthly_trends = create_monthly_trends(
    monthly_kpi
)

display(monthly_trends.head())

,report_month,total_campaigns,total_marketing_spend,total_impressions,total_clicks,total_conversions,new_customers,total_revenue,average_order_value,ctr_percentage,...,new_customers_mom_pct,total_revenue_mom_pct,customer_acquisition_cost_mom_pct,roas_mom_pct,cost_per_click_mom_pct,average_order_value_mom_pct,month_name,revenue_direction,roas_direction,cac_direction
0,2025-01-01,17,1345125.0,6280197,188767,2899,647,4576783.97,1810.88,3.01,...,NaN,NaN,NaN,NaN,NaN,NaN,Jan 2025,No Previous Month,No Previous Month,No Previous Month
1,2025-02-01,17,1310764.0,6288003,222402,3013,681,5064629.74,1888.36,3.54,...,5.26,10.66,-7.42,13.53,-17.39,4.28,Feb 2025,Improved,Improved,Improved
2,2025-03-01,17,1314868.0,6778778,252586,3039,692,5096502.64,1912.32,3.73,...,1.62,0.63,-1.28,0.52,-11.54,1.27,Mar 2025,Improved,Improved,Improved
3,2025-04-01,17,1178049.0,7419426,241276,2953,663,4796561.88,1840.19,3.25,...,-4.19,-5.89,-6.49,4.90,-6.33,-3.77,Apr 2025,Declined,Improved,Improved
4,2025-05-01,17,917719.0,5010443,185918,3004,718,4972700.54,1881.17,3.71,...,8.30,3.67,-28.07,33.17,1.23,2.23,May 2025,Improved,Improved,Improved


In [112]:
def create_quality_report(
    campaigns,
    customers,
    conversions
):
    results = []

    def add_check(
        dataset,
        check_name,
        issue_count,
        comments
    ):
        results.append({
            "dataset": dataset,
            "check_name": check_name,
            "issue_count": int(issue_count),
            "status": (
                "Passed"
                if issue_count == 0
                else "Failed"
            ),
            "comments": comments
        })

    add_check(
        "campaigns",
        "Duplicate Campaign IDs",
        campaigns["Campaign_ID"].duplicated().sum(),
        "Campaign_ID should be unique."
    )

    add_check(
        "campaigns",
        "Missing Campaign Channels",
        campaigns["Channel"].isna().sum(),
        "Channel should not be missing."
    )

    add_check(
        "campaigns",
        "Clicks Greater Than Impressions",
        (
            campaigns["Clicks"]
            > campaigns["Impressions"]
        ).sum(),
        "Clicks should not exceed impressions."
    )

    add_check(
        "customers",
        "Duplicate Customer IDs",
        customers["Customer_ID"].duplicated().sum(),
        "Customer_ID should be unique."
    )

    add_check(
        "customers",
        "Missing Customer Segments",
        customers["Customer_Segment"].isna().sum(),
        "Customer segment should not be missing."
    )

    add_check(
        "customers",
        "Negative Lifetime Value",
        (
            customers["Lifetime_Value"] < 0
        ).sum(),
        "Lifetime value cannot be negative."
    )

    add_check(
        "conversions",
        "Duplicate Conversion IDs",
        conversions["Conversion_ID"].duplicated().sum(),
        "Conversion_ID should be unique."
    )

    add_check(
        "conversions",
        "Missing Payment Methods",
        conversions["Payment_Method"].isna().sum(),
        "Payment method should not be missing."
    )

    add_check(
        "conversions",
        "Negative Revenue",
        (
            conversions["Revenue"] < 0
        ).sum(),
        "Revenue cannot be negative."
    )

    valid_campaign_ids = set(
        campaigns["Campaign_ID"].dropna()
    )

    add_check(
        "conversions",
        "Invalid Campaign References",
        (
            ~conversions["Campaign_ID"].isin(
                valid_campaign_ids
            )
        ).sum(),
        "Campaign_ID must exist in campaigns."
    )

    valid_customer_ids = set(
        customers["Customer_ID"].dropna()
    )

    add_check(
        "conversions",
        "Invalid Customer References",
        (
            ~conversions["Customer_ID"].isin(
                valid_customer_ids
            )
        ).sum(),
        "Customer_ID must exist in customers."
    )

    report = pd.DataFrame(results)

    report["check_date"] = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    logging.info(
        f"Data-quality report created with "
        f"{len(report)} checks."
    )

    return report

In [113]:
data_quality_report = create_quality_report(
    raw_campaigns,
    raw_customers,
    raw_conversions
)

display(data_quality_report)

,dataset,check_name,issue_count,status,comments,check_date
0,campaigns,Duplicate Campaign IDs,0,Passed,Campaign_ID should be unique.,2026-06-22 11:51:33
1,campaigns,Missing Campaign Channels,1,Failed,Channel should not be missing.,2026-06-22 11:51:33
2,campaigns,Clicks Greater Than Impressions,1,Failed,Clicks should not exceed impressions.,2026-06-22 11:51:33
3,customers,Duplicate Customer IDs,0,Passed,Customer_ID should be unique.,2026-06-22 11:51:33
4,customers,Missing Customer Segments,1,Failed,Customer segment should not be missing.,2026-06-22 11:51:33
5,customers,Negative Lifetime Value,1,Failed,Lifetime value cannot be negative.,2026-06-22 11:51:33
6,conversions,Duplicate Conversion IDs,5,Failed,Conversion_ID should be unique.,2026-06-22 11:51:33
7,conversions,Missing Payment Methods,1,Failed,Payment method should not be missing.,2026-06-22 11:51:33
8,conversions,Negative Revenue,1,Failed,Revenue cannot be negative.,2026-06-22 11:51:33
9,conversions,Invalid Campaign References,1,Failed,Campaign_ID must exist in campaigns.,2026-06-22 11:51:33


In [114]:
pipeline_summary = pd.DataFrame({
    "metric": [
        "Pipeline Run Date",
        "Monthly Records",
        "Channels Analysed",
        "Quality Checks",
        "Passed Checks",
        "Failed Checks",
        "Total Issues"
    ],
    "value": [
        datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        len(monthly_trends),
        len(channel_performance),
        len(data_quality_report),
        (
            data_quality_report["status"] == "Passed"
        ).sum(),
        (
            data_quality_report["status"] == "Failed"
        ).sum(),
        data_quality_report["issue_count"].sum()
    ]
})

display(pipeline_summary)

,metric,value
0,Pipeline Run Date,2026-06-22 11:51:42
1,Monthly Records,12
2,Channels Analysed,7
3,Quality Checks,11
4,Passed Checks,2
5,Failed Checks,9
6,Total Issues,13


In [115]:
monthly_trends.to_csv(
    output_folder / "monthly_trends.csv",
    index=False
)

channel_performance.to_csv(
    output_folder / "channel_performance.csv",
    index=False
)

data_quality_report.to_csv(
    output_folder / "data_quality_report.csv",
    index=False
)

pipeline_summary.to_csv(
    output_folder / "pipeline_summary.csv",
    index=False
)

logging.info(
    "All reporting files exported successfully."
)

print("All reporting files exported successfully.")

All reporting files exported successfully.


In [116]:
logging.info(
    "Monthly reporting pipeline completed successfully."
)

print("Pipeline completed successfully.")
print("Log file:", log_file)

Pipeline completed successfully.
Log file: /content/logs/reporting_pipeline.log


In [117]:
latest_month = monthly_trends.sort_values(
    "report_month"
).iloc[-1]

latest_month

,11
report_month,2025-12-01 00:00:00
total_campaigns,16
total_marketing_spend,1406389.0
total_impressions,5532626
total_clicks,514113
total_conversions,2905
new_customers,675
total_revenue,4824669.84
average_order_value,1889.58
ctr_percentage,9.29


In [118]:
best_roas_channel = channel_performance.loc[
    channel_performance["roas"].idxmax()
]

highest_revenue_channel = channel_performance.loc[
    channel_performance["total_revenue"].idxmax()
]

lowest_roas_channel = channel_performance.loc[
    channel_performance["roas"].idxmin()
]

In [119]:
insight_data = {
    "month": latest_month["month_name"],
    "revenue": latest_month["total_revenue"],
    "revenue_mom": latest_month["total_revenue_mom_pct"],
    "spend": latest_month["total_marketing_spend"],
    "spend_mom": latest_month["total_marketing_spend_mom_pct"],
    "conversions": latest_month["total_conversions"],
    "conversions_mom": latest_month["total_conversions_mom_pct"],
    "roas": latest_month["roas"],
    "roas_mom": latest_month["roas_mom_pct"],
    "cac": latest_month["customer_acquisition_cost"],
    "cac_mom": latest_month[
        "customer_acquisition_cost_mom_pct"
    ],
    "best_roas_channel": best_roas_channel["channel"],
    "best_roas": best_roas_channel["roas"],
    "top_revenue_channel": highest_revenue_channel["channel"],
    "top_channel_revenue": highest_revenue_channel[
        "total_revenue"
    ],
    "weakest_channel": lowest_roas_channel["channel"],
    "weakest_roas": lowest_roas_channel["roas"]
}

In [120]:
ai_prompt = f"""
Act as a senior marketing data analyst.

Using the KPI information below, write a concise monthly
performance commentary.

Reporting month: {insight_data['month']}

Monthly KPIs:
- Revenue: ₹{insight_data['revenue']:,.2f}
- Revenue MoM change: {insight_data['revenue_mom']}%
- Marketing spend: ₹{insight_data['spend']:,.2f}
- Spend MoM change: {insight_data['spend_mom']}%
- Conversions: {insight_data['conversions']:,.0f}
- Conversions MoM change: {insight_data['conversions_mom']}%
- ROAS: {insight_data['roas']}
- ROAS MoM change: {insight_data['roas_mom']}%
- CAC: ₹{insight_data['cac']:,.2f}
- CAC MoM change: {insight_data['cac_mom']}%

Channel performance:
- Highest ROAS channel:
  {insight_data['best_roas_channel']}
  with ROAS {insight_data['best_roas']}

- Highest revenue channel:
  {insight_data['top_revenue_channel']}
  with revenue ₹{insight_data['top_channel_revenue']:,.2f}

- Weakest ROAS channel:
  {insight_data['weakest_channel']}
  with ROAS {insight_data['weakest_roas']}

Provide:
1. Executive summary
2. Three key findings
3. Two actionable recommendations

Do not invent reasons that are not supported by the data.
Keep the response under 180 words.
"""

In [121]:
print(ai_prompt)


Act as a senior marketing data analyst.

Using the KPI information below, write a concise monthly
performance commentary.

Reporting month: Dec 2025

Monthly KPIs:
- Revenue: ₹4,824,669.84
- Revenue MoM change: 9.11%
- Marketing spend: ₹1,406,389.00
- Spend MoM change: 26.56%
- Conversions: 2,905
- Conversions MoM change: 5.67%
- ROAS: 3.43
- ROAS MoM change: -13.82%
- CAC: ₹2,083.54
- CAC MoM change: 18.87%

Channel performance:
- Highest ROAS channel:
  Referral
  with ROAS 11.44

- Highest revenue channel:
  Paid Search
  with revenue ₹15,168,287.95

- Weakest ROAS channel:
  Paid Search
  with ROAS 2.52

Provide:
1. Executive summary
2. Three key findings
3. Two actionable recommendations

Do not invent reasons that are not supported by the data.
Keep the response under 180 words.



In [122]:
def change_text(value, metric_name, lower_is_better=False):
    if pd.isna(value):
        return f"No previous-month comparison is available for {metric_name}."

    if value > 0:
        direction = "increased"
    elif value < 0:
        direction = "decreased"
    else:
        return f"{metric_name} remained unchanged."

    impact = ""

    if lower_is_better:
        impact = "an improvement" if value < 0 else "a decline"
    else:
        impact = "an improvement" if value > 0 else "a decline"

    return (
        f"{metric_name} {direction} by {abs(value):.2f}%, "
        f"indicating {impact}."
    )

In [123]:
executive_summary = (
    f"In {insight_data['month']}, the business generated "
    f"₹{insight_data['revenue']:,.2f} in revenue from "
    f"₹{insight_data['spend']:,.2f} in marketing spend, "
    f"resulting in a ROAS of {insight_data['roas']:.2f}."
)

revenue_insight = change_text(
    insight_data["revenue_mom"],
    "Revenue"
)

conversion_insight = change_text(
    insight_data["conversions_mom"],
    "Conversions"
)

cac_insight = change_text(
    insight_data["cac_mom"],
    "Customer acquisition cost",
    lower_is_better=True
)

channel_insight = (
    f"{insight_data['best_roas_channel']} was the most efficient "
    f"channel with a ROAS of {insight_data['best_roas']:.2f}, "
    f"while {insight_data['weakest_channel']} recorded the lowest "
    f"ROAS of {insight_data['weakest_roas']:.2f}."
)

recommendation = (
    f"Consider increasing investment in "
    f"{insight_data['best_roas_channel']} while reviewing campaign "
    f"targeting, bidding, and creative performance for "
    f"{insight_data['weakest_channel']}."
)

monthly_commentary = (
    executive_summary
    + "\n\n"
    + revenue_insight
    + "\n"
    + conversion_insight
    + "\n"
    + cac_insight
    + "\n"
    + channel_insight
    + "\n\nRecommendation: "
    + recommendation
)

print(monthly_commentary)

In Dec 2025, the business generated ₹4,824,669.84 in revenue from ₹1,406,389.00 in marketing spend, resulting in a ROAS of 3.43.

Revenue increased by 9.11%, indicating an improvement.
Conversions increased by 5.67%, indicating an improvement.
Customer acquisition cost increased by 18.87%, indicating a decline.
Referral was the most efficient channel with a ROAS of 11.44, while Paid Search recorded the lowest ROAS of 2.52.

Recommendation: Consider increasing investment in Referral while reviewing campaign targeting, bidding, and creative performance for Paid Search.


In [124]:
commentary_file = output_folder / "monthly_insight_commentary.txt"

with open(
    commentary_file,
    "w",
    encoding="utf-8"
) as file:
    file.write(monthly_commentary)

logging.info(
    "Monthly insight commentary generated successfully."
)

print("Commentary file created:", commentary_file)

Commentary file created: /content/outputs/monthly_insight_commentary.txt


In [125]:
commentary_df = pd.DataFrame({
    "report_month": [latest_month["report_month"]],
    "month_name": [latest_month["month_name"]],
    "executive_summary": [executive_summary],
    "revenue_insight": [revenue_insight],
    "conversion_insight": [conversion_insight],
    "cac_insight": [cac_insight],
    "channel_insight": [channel_insight],
    "recommendation": [recommendation]
})

commentary_df.to_csv(
    output_folder / "monthly_insight_commentary.csv",
    index=False
)

display(commentary_df)

,report_month,month_name,executive_summary,revenue_insight,conversion_insight,cac_insight,channel_insight,recommendation
0,2025-12-01,Dec 2025,"In Dec 2025, the business generated ₹4,824,669...","Revenue increased by 9.11%, indicating an impr...","Conversions increased by 5.67%, indicating an ...","Customer acquisition cost increased by 18.87%,...",Referral was the most efficient channel with a...,Consider increasing investment in Referral whi...
